# Week 7 — Model Training

**File:** `03_model_training_week7.ipynb`

This notebook prepares the processed IPL dataset and trains five regression approaches:
Random Forest, XGBoost, MLP, Hybrid Ensemble, and LightGBM. Each fitted pipeline is saved to `models/`, while training/validation results are saved to `results/`.

> Run this notebook before the Week 8 evaluation notebook.

## 1. Imports

In [ ]:
# Install missing packages only when required:
# !pip install pandas numpy matplotlib scikit-learn xgboost lightgbm joblib

import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports completed.")

In [ ]:
from pathlib import Path

# Find the repository root whether the notebook is opened from /notebooks or the project root.
CURRENT = Path.cwd().resolve()
PROJECT_ROOT = CURRENT.parent if CURRENT.name == "notebooks" else CURRENT
if not (PROJECT_ROOT / "data" / "processed" / "ipl_features.csv").exists():
    candidates = list(CURRENT.rglob("data/processed/ipl_features.csv"))
    if not candidates:
        raise FileNotFoundError("Could not find data/processed/ipl_features.csv")
    PROJECT_ROOT = candidates[0].parents[2]

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "ipl_features.csv"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
PLOTS_DIR = RESULTS_DIR / "plots"

for folder in [MODELS_DIR, RESULTS_DIR, PLOTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATA_PATH}")

## 2. Load processed dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
display(df.head())
print("Columns:", df.columns.tolist())

## 3. Feature preparation

In [ ]:
TARGET = "overall_performance_score"

# Exclude identifiers, text columns, the target, and engineered columns that directly contribute
# to the target score. This reduces target leakage.
EXCLUDE_COLUMNS = {
    TARGET,
    "performance_category",
    "player_id",
    "highest_score",
    "best_bowling",
    "batting_impact",
    "bowling_impact",
    "consistency_score",
    "_outlier",
}

feature_columns = [
    column for column in df.select_dtypes(include=np.number).columns
    if column not in EXCLUDE_COLUMNS
]

model_data = df[feature_columns + [TARGET]].replace([np.inf, -np.inf], np.nan)
model_data = model_data.dropna(subset=[TARGET]).reset_index(drop=True)

X = model_data[feature_columns]
y = model_data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f"Features used: {len(feature_columns)}")
print(feature_columns)
print(f"Training rows: {len(X_train)}")
print(f"Held-out test rows: {len(X_test)}")

In [ ]:
# Before selecting the features perform some basic feature selection using correlation matrix 
# because if dropped features have high correlation with target variables then there could be loss of information.
# so better to drop features which have low correlation with target variables .

In [ ]:
# Better to split data into train, test and validation sets rather than just train and test because validation data helps to test the model on unseen data and helps to tune hyperparameters and avoid overfitting.

In [ ]:
# Save metadata so Week 8 can reproduce the same evaluation setup.
metadata = {
    "target": TARGET,
    "features": feature_columns,
    "random_state": RANDOM_STATE,
    "test_size": 0.20,
    "dataset": str(DATA_PATH.relative_to(PROJECT_ROOT)),
}
with open(RESULTS_DIR / "training_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Saved results/training_metadata.json")

## 4. Define model pipelines

In [ ]:
def numeric_pipeline(model, scale=False):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

random_forest = numeric_pipeline(
    RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
)

xgboost = numeric_pipeline(
    XGBRegressor(
        n_estimators=350,
        learning_rate=0.04,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
)

mlp = numeric_pipeline(
    MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        learning_rate_init=0.001,
        max_iter=700,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=RANDOM_STATE,
    ),
    scale=True,
)

lightgbm = numeric_pipeline(
    LGBMRegressor(
        n_estimators=350,
        learning_rate=0.04,
        num_leaves=31,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )
)

# The hybrid model combines diverse learners. VotingRegressor trains each component
# and averages their predictions using the specified weights.
hybrid = VotingRegressor(
    estimators=[
        ("rf", random_forest),
        ("xgb", xgboost),
        ("mlp", mlp),
        ("lgbm", lightgbm),
    ],
    weights=[2, 2, 1, 2],
    n_jobs=-1,
)

models = {
    "random_forest": random_forest,
    "xgboost": xgboost,
    "mlp": mlp,
    "hybrid": hybrid,
    "lightgbm": lightgbm,
}

print("Models defined:", list(models))

In [ ]:
# Better to perform hyperparameter tuning using GridSearchCV or RandomizedSearchCV to find the best parameters for each model. This can significantly improve model performance.
# using early stopping for the models can save time and help to avoid overfitting.

## 5. Train and save all models

In [ ]:
training_results = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    metrics = {
        "model": name,
        "r2": r2_score(y_test, predictions),
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": mean_squared_error(y_test, predictions) ** 0.5,
    }
    training_results.append(metrics)

    model_path = MODELS_DIR / f"{name}.pkl"
    joblib.dump(model, model_path)
    print(f"Saved: {model_path.name}")

training_results_df = pd.DataFrame(training_results).sort_values("rmse").reset_index(drop=True)
display(training_results_df)

In [ ]:
training_results_df.to_csv(RESULTS_DIR / "week7_training_metrics.csv", index=False)
with open(RESULTS_DIR / "week7_training_metrics.json", "w", encoding="utf-8") as file:
    json.dump(training_results_df.to_dict(orient="records"), file, indent=2)

print("Saved training metrics as CSV and JSON.")

## 6. Quick validation comparison

In [ ]:
ax = training_results_df.sort_values("rmse", ascending=False).plot(
    x="model", y="rmse", kind="bar", legend=False, figsize=(9, 5)
)
ax.set_title("Week 7 Validation RMSE by Model")
ax.set_xlabel("Model")
ax.set_ylabel("RMSE (lower is better)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plot_path = PLOTS_DIR / "week7_validation_rmse.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {plot_path}")

## 7. Week 7 completion check

The following files should now exist:

- `models/random_forest.pkl`
- `models/xgboost.pkl`
- `models/mlp.pkl`
- `models/hybrid.pkl`
- `models/lightgbm.pkl`
- `results/training_metadata.json`
- `results/week7_training_metrics.csv`
- `results/week7_training_metrics.json`
- `results/plots/week7_validation_rmse.png`

In [ ]:
expected = [MODELS_DIR / f"{name}.pkl" for name in models]
expected += [RESULTS_DIR / "training_metadata.json", RESULTS_DIR / "week7_training_metrics.csv"]
for path in expected:
    print("✓" if path.exists() else "✗", path.relative_to(PROJECT_ROOT))

In [ ]:
# over all good model usage but if focused on performing minor changes like performing features selction using correlation matrix 
# then applying hyperparameter tuning using GridSearchCV or RandomizedSearchCV to find the best parameters for each model. This can significantly improve model performance
# and at last if early stopping is used lot of time can be saved and overfitting can be avoided.